In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pywt

## Config

In [4]:
IN_PATH  = "../data/NF-UNSW-NB15-v3.csv"
label_col = 'Attack'
wavelet_name = 'cmor3.5-1.0'
max_samples = 2000
scales = np.arange(1, 128)

In [3]:
#Load the data
df = pd.read_csv(IN_PATH)

In [5]:
attack_types = df["Attack"].unique()

In [6]:
rows = []

In [7]:
for attack_type in attack_types:
    df_attack = df[df["Attack"] == attack_type]
    
    n_samples = min(len(df_attack), max_samples)
    df_attack = df_attack.iloc[:n_samples]
    
    df_attack = df_attack.sort_values(by="FLOW_START_MILLISECONDS")
    
    iat_signal = df_attack["SRC_TO_DST_IAT_AVG"].replace([np.inf, -np.inf], np.nan).dropna().values
    
    if len(iat_signal) == 0:
        continue
    
    iat_signal = (iat_signal - np.mean(iat_signal)) / np.std(iat_signal)
    
    coefficients, frequencies = pywt.cwt(iat_signal, scales, wavelet_name)
    
    flat_coeffs = coefficients.flatten()
    
    row_dict = {"attack_type": attack_type}
    for i, val in enumerate(flat_coeffs):
        row_dict[f"cwt_{i}"] = val
    
    rows.append(row_dict)

In [8]:
df_cwt = pd.DataFrame(rows)

In [9]:
df_cwt.to_csv("cwt_iat_avg_all_attacks.csv", index=False)